In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

df = pd.read_csv("../data/kepler_eda_clean.csv")
target = "koi_disposition"
features = [
    "koi_period",
    "koi_duration",
    "koi_depth",
    "koi_impact",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr"
]

X = df[features]

In [3]:
y = df["koi_disposition"].map({
    "CONFIRMED": 1,
    "FALSE POSITIVE": 0
})

In [4]:
RANDOM_STATE=40

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Train: (5861, 8) | Test: (1466, 8)
Train class balance:
 koi_disposition
0    0.62532
1    0.37468
Name: proportion, dtype: float64
Test class balance:
 koi_disposition
0    0.625512
1    0.374488
Name: proportion, dtype: float64


In [8]:
log_features = ["koi_period", "koi_duration", "koi_depth", "koi_prad",
                 "koi_teq", "koi_insol", "koi_model_snr"]
no_log_features = ["koi_impact"]

log_pipeline = Pipeline(steps=[
    ("log1p", FunctionTransformer(np.log1p, validate=True)),
    ("scale", StandardScaler())
])

no_log_pipeline = Pipeline(steps=[
    ("scale", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("log", log_pipeline, log_features),
    ("nolog", no_log_pipeline, no_log_features),
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...), ('nolog', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name

In [9]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

feature_names_out = log_features + no_log_features

X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names_out, index=X_train.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=feature_names_out, index=X_test.index)

X_train_proc.describe().T

,count,mean,std,min,25%,50%,75%,max
koi_period,5861.0,-3.333889e-17,1.000085,-1.417002,-0.822795,-0.177284,0.563247,2.828501
koi_duration,5861.0,4.000667e-17,1.000085,-2.559171,-0.688537,-0.143116,0.502280,5.286815
koi_depth,5861.0,5.613057e-16,1.000085,-2.368240,-0.681999,-0.274510,0.345753,2.930202
koi_prad,5861.0,-9.698587e-18,1.000085,-1.322127,-0.739338,-0.482469,0.848860,7.061059
koi_teq,5861.0,-7.698253e-16,1.000085,-3.314731,-0.658583,0.028095,0.702860,4.022163
koi_insol,5861.0,-2.606495e-17,1.000085,-1.982100,-0.716504,-0.002050,0.709688,4.217384
koi_model_snr,5861.0,2.121566e-16,1.000085,-2.059754,-0.724013,-0.323692,0.472576,3.062997
koi_impact,5861.0,1.212323e-17,1.000085,-0.224303,-0.157817,-0.054588,0.042226,29.171448


In [12]:
assert not X_train_proc.isna().any().any(), "NaN in X_train_proc!"
assert not X_test_proc.isna().any().any(), "NaN in X_test_proc!"
assert np.isfinite(X_train_proc.values).all(), "Inf in X_train_proc!"
assert np.isfinite(X_test_proc.values).all(), "Inf in X_test_proc!"

print("Mean (train, should be around ~0):")
print(X_train_proc.mean().round(3))
print("std (train, should be around ~1):")
print(X_train_proc.std().round(3))

Mean (train, should be around ~0):
koi_period      -0.0
koi_duration     0.0
koi_depth        0.0
koi_prad        -0.0
koi_teq         -0.0
koi_insol       -0.0
koi_model_snr    0.0
koi_impact       0.0
dtype: float64
std (train, should be around ~1):
koi_period       1.0
koi_duration     1.0
koi_depth        1.0
koi_prad         1.0
koi_teq          1.0
koi_insol        1.0
koi_model_snr    1.0
koi_impact       1.0
dtype: float64


In [16]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

X_train_proc.to_csv(processed_dir / "X_train.csv", index=False)
X_test_proc.to_csv(processed_dir / "X_test.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

# raw for tree models
X_train.to_csv(processed_dir / "X_train_raw.csv", index=False)
X_test.to_csv(processed_dir / "X_test_raw.csv", index=False)


print("Saved to:", processed_dir.resolve())
list(processed_dir.iterdir())

Saved to: C:\Users\kamil\PythonProjects\exoplanet_classification\data\processed


[WindowsPath('../data/processed/X_test.csv'),
 WindowsPath('../data/processed/X_test_raw.csv'),
 WindowsPath('../data/processed/X_train.csv'),
 WindowsPath('../data/processed/X_train_raw.csv'),
 WindowsPath('../data/processed/y_test.csv'),
 WindowsPath('../data/processed/y_train.csv')]